<a href="https://colab.research.google.com/github/Engr-Muhammad-Anees/Dubbing-Podcast-ML/blob/main/dp_podcasting123.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**install modules:**

In [ ]:
!pip install moviepy
!pip install gtts
!pip install pydub
!apt-get install -y ffmpeg
!pip install assemblyai

**import libraries:**

In [4]:
import assemblyai as aai
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip
from gtts import gTTS
from pydub import AudioSegment
from pydub.effects import speedup
import os
import tempfile
import json
from typing import List, Dict

**input audio:**

In [8]:
audio_path = "/content/audio_normalized.wav"

**assemblyai setup:**

In [ ]:
aai.settings.api_key = "YOUR_ASSEMBLYAI_API_KEY"  # Set your AssemblyAI API key

def transcribe_with_diarization(audio_path: str) -> List[Dict]:
    """Transcribe audio with speaker diarization using AssemblyAI
    Args:
        audio_path (str): Path to the audio file to transcribe
    Returns:
        List[Dict]: List of utterances with speaker labels, text, and timestamps
    """
    config = aai.TranscriptionConfig(
        speaker_labels=True,      # Enable speaker diarization
        speakers_expected=2       # Expected number of speakers
    )

    transcriber = aai.Transcriber() # Initialize the transcriber that will handle the API communication
    transcript = transcriber.transcribe(audio_path, config) # Send the audio path and it performs both speech to text and speaker diarization in one API call

    if transcript.status == aai.TranscriptStatus.error
        raise Exception(f"Transcription failed: {transcript.error}")  # If there was an error

    utterances = [] #empty list to store all the processed utterances
    for utterance in transcript.utterances:
        utterances.append({
            'speaker': f"Speaker_{utterance.speaker}",  # Convert to format like "Speaker_A", "Speaker_B"
            'text': utterance.text,                      # The transcribed text of what was said
            'start': utterance.start,                    # Start time in milliseconds
            'end': utterance.end,                        # End time in milliseconds
            'duration': utterance.end - utterance.start  # Calculate duration in milliseconds
        })

    return utterances

**Text-to-Speech for Dubbing:**

In [ ]:
def create_dubbed_audio(utterances: List[Dict], target_language: str = 'en') -> str:
    """Create dubbed audio using gTTS for each speaker segment"""
    base_audio = AudioSegment.silent(duration=utterances[-1]['end'] + 1000)  # Create silent audio of total duration + buffer

    # Map speakers to consistent voices
    speaker_voices = {}
    available_voices = {
        'en': ['com', 'us'],  # English voices
    }

    for i, utterance in enumerate(utterances):
        print(f"Processing utterance {i+1}/{len(utterances)}: {utterance['speaker']}")

        speaker = utterance['speaker']# Assign unique voice to each speaker
        if speaker not in speaker_voices
            voice_idx = len(speaker_voices) % len(available_voices.get(target_language, ['com']))
            speaker_voices[speaker] = available_voices.get(target_language, ['com'])[voice_idx]

        try:
            tts = gTTS(                                # Generate text-to-speech audio
                text=utterance['text'],
                lang=target_language,
                slow=False,
                tld=speaker_voices[speaker]  # Use assigne voice
            )

            # Save TTS to temp file
            with tempfile.NamedTemporaryFile(delete=False, suffix='.mp3') as temp_tts:
                tts.save(temp_tts.name)
                tts_audio = AudioSegment.from_mp3(temp_tts.name)   # Load generated audio

                original_duration = utterance['duration']   # Adjust timing to match original
                tts_duration = len(tts_audio)

                if tts_duration > original_duration:
                    speed_factor = tts_duration / original_duration
                    tts_audio = speedup(tts_audio, playback_speed=speed_factor)

                start_time = utterance['start'] # Place audio at correct timestamp
                base_audio = base_audio.overlay(tts_audio, position=start_time)

            os.unlink(temp_tts.name)

        except Exception as e:
            print(f"Error processing utterance {i+1}: {e}")
            continue

    output_path = "/content/dubbed_audio.wav"    # Export final dubbed audio
    base_audio.export(output_path, format="wav")

    return output_path

**display dudded_audio**

In [24]:
dubbed_audio = '/content/dubbed_audio.wav'

In [25]:
from IPython.display import Audio
Audio(dubbed_audio)

**merged input video with dubbing audio and get dubbed_video output:**

In [ ]:
!ffmpeg -i input_video_one.mp4 -i dubbed_audio.wav -c:v copy -map 0:v:0 -map 1:a:0 -shortest final_dubbed_video.mp4


In [27]:
from IPython.display import Video
Video("/content/final_dubbed_video.mp4")